## Construction du Pipeline Complet pour Streaming

Pour permettre au streaming de traiter les données brutes (avec variables catégorielles), nous devons créer un pipeline complet qui inclut l'encodage.

In [8]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gestionlogistique").getOrCreate()
df = spark.read.option("header", "true").parquet("data/output_training_parquet/part-00000-97d86afc-357a-4d4f-96af-144a5770788b-c000.snappy.parquet")

df.show()
df.printSchema()

+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----------------+--------------------+-----------------------+
|Benefit per order|Sales per customer|Late_delivery_risk|Order Item Quantity|Order Item Product Price|Order Item Discount|  Order Item Total|Order Profit Per Order|          distance|Order Country_ohe|     Type_ohe|Customer State_ohe|Order Region_ohe|Shipping Mode_ohe|Department Name_ohe|Category Name_ohe|    numeric_features|scaled_numeric_features|
+-----------------+------------------+------------------+-------------------+------------------------+-------------------+------------------+----------------------+------------------+-----------------+-------------+------------------+----------------+-----------------+-------------------+-----

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.sql.functions import col

# Load RAW data (before encoding) from CSV
df = spark.read.option("header", "true").csv(
    "data/output_tmp/part-00000-039c67c4-4607-444e-be84-ab46ccdf438b-c000.csv"
)

# 1) CAST NUMERIC COLUMNS TO DOUBLE
numeric_columns = [
    "Benefit per order",
    "Sales per customer",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Total",
    "Order Profit Per Order",
    "distance"
]

for c in numeric_columns:
    df = df.withColumn(c, col(c).cast("double"))

# Cast label to integer
df = df.withColumn("Late_delivery_risk", col("Late_delivery_risk").cast("int"))

# Drop rows with missing values
df = df.na.drop(subset=numeric_columns + ["Late_delivery_risk"])

# 2) DROP UNNECESSARY COLUMNS
df = df.drop("Order State", "Days for shipping (real)", 
             "Days for shipment (scheduled)", "Delivery Status")

# 3) CATEGORICAL COLUMNS (NO Customer State)
categorical_columns = [
    "Order Country", "Type",
    "Order Region", "Shipping Mode",
    "Department Name", "Category Name"
]

# Ensure categorical columns are STRING
for c in categorical_columns:
    df = df.withColumn(c, col(c).cast("string"))

stages = []

# 4) STRING INDEXER + ONE-HOT ENCODING
for col_name in categorical_columns:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )
    stages.append(indexer)

    encoder = OneHotEncoder(
        inputCol=col_name + "_index",
        outputCol=col_name + "_ohe"
    )
    stages.append(encoder)

# 5) VECTOR ASSEMBLER
feature_cols = numeric_columns + [c + "_ohe" for c in categorical_columns]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
stages.append(assembler)


gbt = GBTClassifier(
    labelCol="Late_delivery_risk",
    featuresCol="features",
    maxIter=150,
    maxDepth=8,
    stepSize=0.05,
    subsamplingRate=0.8,
    seed=42
)
stages.append(gbt)


pipeline = Pipeline(stages=stages)

for i, stage in enumerate(stages):
    print(f"  {i+1}. {stage.__class__.__name__}")

model = pipeline.fit(df)


path = "models/gbt_pipeline_model"
model.write().overwrite().save(path)

print(f"\n✅ Model saved to {path}")

🔄 Training complete pipeline...
Pipeline has 14 stages:
  1. StringIndexer
  2. OneHotEncoder
  3. StringIndexer
  4. OneHotEncoder
  5. StringIndexer
  6. OneHotEncoder
  7. StringIndexer
  8. OneHotEncoder
  9. StringIndexer
  10. OneHotEncoder
  11. StringIndexer
  12. OneHotEncoder
  13. VectorAssembler
  14. GBTClassifier


25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1000.5 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1001.1 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1000.5 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1001.1 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1002.4 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1002.4 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1009.0 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1016.0 KiB
25/11/20 22:29:36 WARN DAGScheduler: Broadcasting large task binary with size 1009.0 KiB
25/11/20 22:29:36 WAR


✅ Model saved to models/gbt_pipeline_model
📌 Ready for streaming inference!
